## difference in time taken on computaion over cpu vs gpu

In [ ]:
import torch
print(torch.__version__)

2.9.0+cu126


In [ ]:
if torch.cuda.is_available():
  device = torch.device("cuda")
  print(torch.cuda.get_device_properties())
else:
  print("No GPU found")

_CudaDeviceProperties(name='Tesla T4', major=7, minor=5, total_memory=15095MB, multi_processor_count=40, uuid=7bf55b0c-af07-7b55-a92a-0c63334b72e6, pci_bus_id=0, pci_device_id=4, pci_domain_id=0, L2_cache_size=4MB)


In [ ]:
s = 10000
mat_A = torch.rand(size=(s,s), dtype=torch.float64)
mat_B = torch.rand(size=(s,s), dtype=torch.float64)

print(mat_A.shape)
print(mat_B.shape)

torch.Size([10000, 10000])
torch.Size([10000, 10000])


In [ ]:
#cpu computation
import time

start = time.time()
result1 = torch.matmul(mat_A, mat_B)
time_taken_cpu = time.time() - start
print(time_taken_cpu)

31.112018823623657


In [ ]:
#gpu computation
mat_A_gpu = mat_A.to(device)
mat_B_gpu = mat_B.to(device)

import time
start = time.time()
result2 = torch.matmul(mat_A_gpu, mat_B_gpu)
time_taken_gpu = time.time() - start
print(time_taken_gpu)

0.0980989933013916


In [ ]:
print(f"{int(time_taken_cpu/time_taken_gpu)} times faster computations")

317 times faster computations


## neural network implementation in pytorch


In [ ]:
import torch

if torch.cuda.is_available():
  device = torch.device("cuda")
  print(torch.cuda.get_device_properties())
else:
  print("No GPU found")

No GPU found


In [ ]:
import torch
import pandas as pd, kagglehub
path = kagglehub.dataset_download("sonalshinde123/daily-water-intake-and-hydration-patterns-dataset")

df = pd.read_csv(f"{path}/Daily_Water_Intake.csv")
df = df.sample(frac=0.2, random_state=42)
df.head()

Using Colab cache for faster access to the 'daily-water-intake-and-hydration-patterns-dataset' dataset.


,Age,Gender,Weight (kg),Daily Water Intake (liters),Physical Activity Level,Weather,Hydration Level
2308,57,Male,70,2.48,Moderate,Cold,Good
22404,39,Male,95,3.93,Moderate,Hot,Good
23397,18,Female,54,2.36,Moderate,Normal,Good
25058,57,Female,61,1.50,Low,Cold,Poor
2664,45,Male,81,3.40,Low,Hot,Good


In [ ]:
df.shape

(6000, 7)

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 6000 entries, 2308 to 29171
Data columns (total 7 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Age                          6000 non-null   int64  
 1   Gender                       6000 non-null   object 
 2   Weight (kg)                  6000 non-null   int64  
 3   Daily Water Intake (liters)  6000 non-null   float64
 4   Physical Activity Level      6000 non-null   object 
 5   Weather                      6000 non-null   object 
 6   Hydration Level              6000 non-null   object 
dtypes: float64(1), int64(2), object(4)
memory usage: 375.0+ KB


In [ ]:
# lets do one-hot encoding for followings - Gender, Physical Activity Level & Weather
df = pd.get_dummies(df, prefix=["Gender", "PAL", "Weather"], columns=["Gender", "Physical Activity Level", "Weather"], dtype=int)
df.head()

,Age,Weight (kg),Daily Water Intake (liters),Hydration Level,Gender_Female,Gender_Male,PAL_High,PAL_Low,PAL_Moderate,Weather_Cold,Weather_Hot,Weather_Normal
2308,57,70,2.48,Good,0,1,0,0,1,1,0,0
22404,39,95,3.93,Good,0,1,0,0,1,0,1,0
23397,18,54,2.36,Good,1,0,0,0,1,0,0,1
25058,57,61,1.50,Poor,1,0,0,1,0,1,0,0
2664,45,81,3.40,Good,0,1,0,1,0,0,1,0


In [ ]:
y = df['Hydration Level']
df = df.drop(columns=["Hydration Level"], axis=1)
X = df[:]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print(X_train.shape)
print(X_test.shape)

(4800, 11)
(1200, 11)


In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:
X_train_t = torch.from_numpy(X_train)
X_test_t = torch.from_numpy(X_test)
y_train_t = torch.from_numpy(y_train)
y_test_t = torch.from_numpy(y_test)

print(X_train_t.shape, y_train_t.shape)

torch.Size([4800, 11]) torch.Size([4800])


In [ ]:
class NeuralNetwork:
  def __init__(self, inputs):
    # initalize weights and biases for the perceptron
    self.weights = torch.rand(size=(1, inputs), dtype=torch.float64, requires_grad=True)
    self.biases = torch.zeros(1, dtype=torch.float64, requires_grad=True)
    # print(self.weights.shape, self.biases.shape)

  def forward_pass(self, X):
    # pass input into the neural network
     self.z = torch.matmul(self.weights, torch.transpose(X, 1, 0)) + self.biases
     self.y_pred = torch.sigmoid(self.z)
    #  print(self.y_pred.shape)
     return self.y_pred

  def loss_computation(self, y):
    # print(self.y_pred.shape, y.shape)
    # Clamp predictions to avoid log(0)
    epsilon = 1e-7
    self.y_pred = torch.clamp(self.y_pred, epsilon, 1 - epsilon)
    # apply binary cross entropy loss here
    l = -(y*torch.log(self.y_pred) + (1-y)*torch.log(1-self.y_pred))
    self.loss = l.mean()
    # print(self.loss.shape)
    return self.loss

  def backward_pass(self):
    self.loss.backward()

  def update_parameters(self, alpha):
    with torch.no_grad():
      w, b = self.weights.grad, self.biases.grad
      # print(w,b)
      self.weights-=alpha*w
      self.biases-=alpha*b
    self.weights.grad.zero_()
    self.biases.grad.zero_()

In [ ]:
alpha = 0.1
epoch = 100

nn = NeuralNetwork(inputs=X_train.shape[1])

for e in range(epoch):
  nn.forward_pass(X_train_t)
  loss = nn.loss_computation(y_train_t)
  nn.backward_pass()
  nn.update_parameters(alpha)

  print(f"Epoch {e+1} & Loss {loss.item()}")


Epoch 1 & Loss 0.7662475280086046
Epoch 2 & Loss 0.7411843674790541
Epoch 3 & Loss 0.7176032365095274
Epoch 4 & Loss 0.6954310857414124
Epoch 5 & Loss 0.6745941574409567
Epoch 6 & Loss 0.655018819274802
Epoch 7 & Loss 0.636632304280655
Epoch 8 & Loss 0.6193633481353764
Epoch 9 & Loss 0.6031427202807566
Epoch 10 & Loss 0.5879036501241344
Epoch 11 & Loss 0.5735821532580535
Epoch 12 & Loss 0.5601172654120031
Epoch 13 & Loss 0.5474511937048689
Epoch 14 & Loss 0.5355293958038623
Epoch 15 & Loss 0.5243005979398518
Epoch 16 & Loss 0.5137167625202859
Epoch 17 & Loss 0.5037330154607735
Epoch 18 & Loss 0.4943075424574916
Epoch 19 & Loss 0.4854014623611355
Epoch 20 & Loss 0.4769786846848222
Epoch 21 & Loss 0.46900575715682086
Epoch 22 & Loss 0.46145170816676967
Epoch 23 & Loss 0.45428788798496045
Epoch 24 & Loss 0.44748781177682456
Epoch 25 & Loss 0.44102700669555467
Epoch 26 & Loss 0.4348828647130474
Epoch 27 & Loss 0.4290345023355964
Epoch 28 & Loss 0.4234626279353721
Epoch 29 & Loss 0.41814941

In [ ]:
from sklearn.metrics import accuracy_score

with torch.no_grad():
  y_pred_t = nn.forward_pass(X_test_t)
  y_pred_t = (y_pred_t > 0.5).float()
  score = accuracy_score(y_test_t, y_pred_t.squeeze())
  print(score)

0.8875


## cleanest pytorch neural network template
1. do feature engineering
2. load train, test datasets - define customdataset and dataloader
2. inherited class from nn.module with __init__, **forward**, **save** and **load** defined
2. define model, loss function, optimizers, learning rate, epochs etc.
4. code training pipeline, code test pipeline
5. save the model

In [ ]:
import torch
import pandas as pd, kagglehub
path = kagglehub.dataset_download("sonalshinde123/daily-water-intake-and-hydration-patterns-dataset")

df = pd.read_csv(f"{path}/Daily_Water_Intake.csv")
# df = df.sample(frac=0.2, random_state=42) # increase training dataset, use mini-batch SGD
df.head()

Using Colab cache for faster access to the 'daily-water-intake-and-hydration-patterns-dataset' dataset.


,Age,Gender,Weight (kg),Daily Water Intake (liters),Physical Activity Level,Weather,Hydration Level
0,56,Male,96,4.23,Moderate,Hot,Good
1,60,Male,105,3.95,High,Normal,Good
2,36,Male,68,2.39,Moderate,Cold,Good
3,19,Female,74,3.13,Moderate,Hot,Good
4,38,Male,77,2.11,Low,Normal,Poor


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 7 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   Age                          30000 non-null  int64  
 1   Gender                       30000 non-null  object 
 2   Weight (kg)                  30000 non-null  int64  
 3   Daily Water Intake (liters)  30000 non-null  float64
 4   Physical Activity Level      30000 non-null  object 
 5   Weather                      30000 non-null  object 
 6   Hydration Level              30000 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 1.6+ MB


In [ ]:
df = pd.get_dummies(df, prefix=["Gender", "PAL", "Weather"], columns=["Gender", "Physical Activity Level", "Weather"], dtype=int)
df.head()

,Age,Weight (kg),Daily Water Intake (liters),Hydration Level,Gender_Female,Gender_Male,PAL_High,PAL_Low,PAL_Moderate,Weather_Cold,Weather_Hot,Weather_Normal
0,56,96,4.23,Good,0,1,0,0,1,0,1,0
1,60,105,3.95,Good,0,1,1,0,0,0,0,1
2,36,68,2.39,Good,0,1,0,0,1,1,0,0
3,19,74,3.13,Good,1,0,0,0,1,0,1,0
4,38,77,2.11,Poor,0,1,0,1,0,0,0,1


In [ ]:
y = df['Hydration Level']
df = df.drop(columns=["Hydration Level"], axis=1)
X = df[:]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, y_train.shape)

(24000, 11) (24000,)


In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

encoder = LabelEncoder()
y_train = encoder.fit_transform(y_train)
y_test = encoder.transform(y_test)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class MyDS(Dataset):
  def __init__(self, features, labels):
    self.X = torch.tensor(features, dtype=torch.float32)
    self.y = torch.tensor(labels, dtype=torch.float32)

  def __len__(self):
    return len(self.X)

  def __getitem__(self, index):
    return self.X[index], self.y[index]


train_DS = MyDS(X_train, y_train)
test_DS = MyDS(X_test, y_test)

train_Loader = DataLoader(train_DS, batch_size=50, shuffle=True)
test_Loader = DataLoader(test_DS, batch_size=50, shuffle=True)

print(len(train_Loader), len(test_Loader))


480 120


In [ ]:
# code the neural network
import torch.nn as nn

class MyNN(nn.Module):
  def __init__(self, input_size):
    super().__init__()
    self.network=nn.Sequential(
        nn.Linear(input_size, 7),
        nn.ReLU(),
        nn.Linear(7, 4),
        nn.ReLU(),
        nn.Linear(4, 1),
        nn.Sigmoid(),
    )

  def forward(self, X):
    return self.network(X)

In [ ]:
# alpha, epochs, model, optimizers, loss

neural_net = MyNN(X_train.shape[1])

alpha, epochs = 0.03, 10
lossfn = nn.BCELoss()
optimizer = torch.optim.SGD(neural_net.parameters(), lr=alpha)

In [ ]:
# training pipeline

def train_neural_net(model, dataLoader):
  model.train()
  for e in range(epochs):
    print(f"E:{e+1}", "-"*50)
    for batch, (X, y) in enumerate(dataLoader):
      # forward pass
      y_pred = model(X)
      # calculate loss
      loss = lossfn(y_pred.squeeze(1), y)
      # zero grads & backward pass
      optimizer.zero_grad()
      loss.backward()
      # update parmas
      optimizer.step()
      # log epoch and loss
      if ((batch+1)%80)==0:
        print(f"Batch: {batch+1} | Loss: {loss.item()}")

In [ ]:
# test pipeline
from sklearn.metrics import accuracy_score

def test_neural_net(model, dataLoader):
  model.eval()
  batch_wise_accuracy = []
  for batch, (X, y) in enumerate(dataLoader):
    # forward pass
    y_pred = model(X)
    y_pred = (y_pred.squeeze(1) > 0.5).float()
    # calculate accuracy
    score = accuracy_score(y, y_pred)
    batch_wise_accuracy.append(score)

  print(batch_wise_accuracy)
  print(f"Average: {sum(batch_wise_accuracy)/len(batch_wise_accuracy)}")

In [ ]:
# hit training and test pipelines
train_neural_net(neural_net, train_Loader)

E:1 --------------------------------------------------
Batch: 80 | Loss: 0.5394073128700256
Batch: 160 | Loss: 0.4554002285003662
Batch: 240 | Loss: 0.44075125455856323
Batch: 320 | Loss: 0.3937395215034485
Batch: 400 | Loss: 0.33121737837791443
Batch: 480 | Loss: 0.3348085880279541
E:2 --------------------------------------------------
Batch: 80 | Loss: 0.2759735584259033
Batch: 160 | Loss: 0.23465953767299652
Batch: 240 | Loss: 0.20739535987377167
Batch: 320 | Loss: 0.2525845468044281
Batch: 400 | Loss: 0.19499975442886353
Batch: 480 | Loss: 0.2158709466457367
E:3 --------------------------------------------------
Batch: 80 | Loss: 0.2639409005641937
Batch: 160 | Loss: 0.21896831691265106
Batch: 240 | Loss: 0.25350794196128845
Batch: 320 | Loss: 0.20070213079452515
Batch: 400 | Loss: 0.1719864457845688
Batch: 480 | Loss: 0.1688026785850525
E:4 --------------------------------------------------
Batch: 80 | Loss: 0.1243334412574768
Batch: 160 | Loss: 0.08158314973115921
Batch: 240 | Lo

In [ ]:
test_neural_net(neural_net, test_Loader)

[1.0, 0.96, 0.98, 1.0, 1.0, 1.0, 0.98, 1.0, 0.96, 1.0, 1.0, 1.0, 1.0, 1.0, 0.98, 1.0, 0.98, 1.0, 1.0, 1.0, 1.0, 1.0, 0.98, 1.0, 1.0, 0.98, 1.0, 0.96, 1.0, 1.0, 1.0, 0.98, 1.0, 0.98, 1.0, 1.0, 1.0, 1.0, 0.98, 0.98, 1.0, 1.0, 1.0, 1.0, 0.98, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.98, 1.0, 1.0, 1.0, 0.98, 1.0, 0.98, 1.0, 1.0, 1.0, 1.0, 1.0, 0.96, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.98, 0.98, 0.98, 0.96, 1.0, 1.0, 0.96, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.98, 1.0, 1.0, 1.0, 0.98, 0.98, 1.0, 0.98, 1.0, 1.0, 1.0, 0.96, 1.0, 1.0, 1.0, 0.98, 0.98, 1.0, 1.0, 1.0, 1.0, 1.0, 0.98, 1.0, 0.98, 1.0]
Average: 0.9934999999999999


## ways to optimize it even further

use **more samples** for training, use a **different optimizer** - different optimization technique, **increase learning rate**, **increase epochs**, use **different weight initialization** techniques, change model architecture - **hyper parameter tuning**

1. l2 regularization
2. dropout - neural network simplified & in every forward pass a different nn is initalized which creates regularization effect during evaluation dropout is turned off an we use the full complex neural network
3. batch normalization

In [ ]:
class MyNN(nn.Module):
    def __init__(self, num_features):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Dropout(p=0.3),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.model(x)

optimizer = optim.SGD(model.parameters(), lr=0.1, weight_decay=1e-4)

NameError: name 'nn' is not defined

## revision - dated - 12 April 2026


In [ ]:
# Install dependencies as needed:
# pip install kagglehub[pandas-datasets]
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "fitness_dataset.csv"

# Load the latest version
df = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "parasharmanu/synthetic-health-indicators-dataset",
  file_path,
  # Provide any additional arguments like
  # sql_query or pandas_kwargs. See the
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

df.head()

/tmp/ipykernel_31274/1369397206.py:10: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  df = kagglehub.load_dataset(


Using Colab cache for faster access to the 'synthetic-health-indicators-dataset' dataset.


,age,steps_per_day,active_minutes,sedentary_minutes,sleep_hours,sleep_quality,stress_level,workouts_per_week,diet_quality,hydration_liters,bmi,heart_rate_resting,heart_rate_avg,calories_burned,consistency_score,lifestyle_category,fitness_level
0,56,12633,122,1084,8.65,8.67,4.42,2,5.61,2.14,16.00,54.8,103.0,1500.0,3.84,Active,Moderate
1,46,1980,33,1122,5.97,5.37,7.91,2,8.17,0.92,37.07,75.2,126.3,1500.0,7.25,Active,Unfit
2,32,7139,60,1049,6.23,7.15,4.20,0,3.47,2.67,27.36,66.3,101.4,1500.0,5.77,Sedentary,Unfit
3,60,4814,66,1018,5.09,7.61,7.12,4,5.26,2.17,24.59,64.6,107.8,1500.0,9.11,Active,Moderate
4,25,8096,86,1031,5.49,5.57,3.88,3,2.60,2.91,19.07,64.0,92.8,1500.0,7.55,Active,Moderate


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 17 columns):
 #   Column              Non-Null Count    Dtype  
---  ------              --------------    -----  
 0   age                 1000000 non-null  int64  
 1   steps_per_day       1000000 non-null  int64  
 2   active_minutes      1000000 non-null  int64  
 3   sedentary_minutes   1000000 non-null  int64  
 4   sleep_hours         1000000 non-null  float64
 5   sleep_quality       1000000 non-null  float64
 6   stress_level        1000000 non-null  float64
 7   workouts_per_week   1000000 non-null  int64  
 8   diet_quality        1000000 non-null  float64
 9   hydration_liters    1000000 non-null  float64
 10  bmi                 1000000 non-null  float64
 11  heart_rate_resting  1000000 non-null  float64
 12  heart_rate_avg      1000000 non-null  float64
 13  calories_burned     1000000 non-null  float64
 14  consistency_score   1000000 non-null  float64
 15  lifestyle_catego

In [ ]:
df["fitness_level"].value_counts()

,count
fitness_level,
Moderate,573629
Unfit,379990
Fit,46381


In [ ]:
# Droping lifestyle_category none of our use
df=df.drop("lifestyle_category", axis=1)
# Define X, y
y = df["fitness_level"]
df=df.drop("fitness_level", axis=1)
X = df[:]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(750000, 15) (250000, 15) (750000,) (250000,)


| Method         | Mean used         | Std used          | Consistent with training? |
| -------------- | ----------------- | ----------------- | ------------------------- |
| fit_transform  | From that dataset | From that dataset | ❌ No                      |
| transform only | From training set | From training set | ✅ Yes                     |


In [ ]:
from sklearn.preprocessing import StandardScaler, LabelEncoder

scaler=StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

encoder=LabelEncoder()
y_train=encoder.fit_transform(y_train)
y_test=encoder.transform(y_test)

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class HealthIndicatorDS(Dataset):
  def __init__(self, X, y):
    self.X = torch.tensor(X, dtype=torch.float32)
    self.y = torch.tensor(y, dtype=torch.long)

  def __len__(self):
    return len(self.X)

  def __getitem__(self, index):
    return self.X[index], self.y[index]

torch_train = HealthIndicatorDS(X_train, y_train)
torch_test = HealthIndicatorDS(X_test, y_test)

torch_train_loader = DataLoader(torch_train, batch_size=750, shuffle=True)
torch_test_loader = DataLoader(torch_test, batch_size=750, shuffle=False)

print(f"""
Total size of train dataset: {len(torch_train)}
Total size of test dataset: {len(torch_test)}
Batch Size: 750
Per Batch train dataset size: {len(torch_train_loader)}
Per Batch test dataset size: {len(torch_test_loader)}
""")


Total size of train dataset: 750000
Total size of test dataset: 250000
Batch Size: 750
Per Batch train dataset size: 1000
Per Batch test dataset size: 334



In [ ]:
# neural network
import torch.nn as nn

class HealthIndicatorNN(nn.Module):
  def __init__(self, input_size):
    super().__init__()
    self.neuralnet = nn.Sequential(
        nn.Linear(input_size, 30),
        nn.BatchNorm1d(30),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(30, 15),
        nn.BatchNorm1d(15),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(15, 10),
        nn.BatchNorm1d(10),
        nn.ReLU(),
        nn.Dropout(p=0.3),
        nn.Linear(10, 3),
    )

  def forward(self, X):
    return self.neuralnet(X)

net = HealthIndicatorNN(X_train.shape[1])
epoch, alpha = 25, 0.01
loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(net.parameters(), lr=alpha)

In [ ]:
# training loop
net.train()
for e in range(epoch):
  print(f"E:{e+1}", "-"*50)
  for batch, (X, y) in enumerate(torch_train_loader):
    # forward pass
    y_pred = net(X)
    # calculate loss
    loss = loss_function(y_pred, y)
    # zero gradient and backward pass
    optimizer.zero_grad()
    loss.backward()
    # update weights
    optimizer.step()
    # -----------------------------------
    if ((batch+1)%150==0):
      print(f"Batch: {batch+1} | Loss: {loss.item()}")

E:1 --------------------------------------------------
Batch: 150 | Loss: 0.4352586567401886
Batch: 300 | Loss: 0.4292660355567932
Batch: 450 | Loss: 0.38706639409065247
Batch: 600 | Loss: 0.439424067735672
Batch: 750 | Loss: 0.36043813824653625
Batch: 900 | Loss: 0.4343826472759247
E:2 --------------------------------------------------
Batch: 150 | Loss: 0.3948793411254883
Batch: 300 | Loss: 0.3709627389907837
Batch: 450 | Loss: 0.37562793493270874
Batch: 600 | Loss: 0.37066903710365295
Batch: 750 | Loss: 0.40654829144477844
Batch: 900 | Loss: 0.37760379910469055
E:3 --------------------------------------------------
Batch: 150 | Loss: 0.3779734671115875
Batch: 300 | Loss: 0.39616551995277405
Batch: 450 | Loss: 0.3752090632915497
Batch: 600 | Loss: 0.3922554850578308
Batch: 750 | Loss: 0.36050957441329956
Batch: 900 | Loss: 0.4146605134010315
E:4 --------------------------------------------------
Batch: 150 | Loss: 0.41227632761001587
Batch: 300 | Loss: 0.37750065326690674
Batch: 450 

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

net.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for X_batch, y_batch in torch_test_loader:
        outputs = net(X_batch)
        _, predicted = torch.max(outputs, 1)
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

import numpy as np
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)

# 🔹 Metrics
accuracy = (all_preds == all_labels).mean()
precision = precision_score(all_labels, all_preds, average='macro')
recall = recall_score(all_labels, all_preds, average='macro')
f1 = f1_score(all_labels, all_preds, average='macro')

# 🔹 Print
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1 Score:  {f1:.4f}")

Accuracy:  0.9128
Precision: 0.9012
Recall:    0.6943
F1 Score:  0.7275
